# TorchRL A2C

In this tutorial, we will learn how to use A2C

This notebook was modified from the official tutorials provided by torchRL: 

**Get started with TorchRL's modules**
Author: `Vincent Moens <https://github.com/vmoens>`
url: https://docs.pytorch.org/rl/stable/tutorials/getting-started-3.html

### Install TorchRL

First let's install torchrl and tensordict.


In [1]:
#!pip install tensordict
#!pip install torchrl

In [2]:
import torchvision 

from tensordict.nn import TensorDictSequential

import torch
from tensordict.nn import TensorDictModule
from tensordict import TensorDict
from torchrl.envs import GymEnv
from torch import nn

from torchrl.modules import QValueActor
from torch.optim import Adam, RMSprop
from torchrl.modules import QValueActor

### PPO

In [3]:


from torchrl.envs import TransformedEnv, Compose, ObservationNorm, DoubleToFloat, StepCounter, RewardScaling
from torchrl.objectives import A2CLoss
from torchrl.objectives.value import TD0Estimator

base_env = GymEnv("Pendulum-v1")

base_env

RewardScaling(loc=0, scale=0.1)  # simple fixed scaling


env = TransformedEnv(
    base_env,
    Compose(
        # normalize observations
        ObservationNorm(in_keys=["observation"]),
        DoubleToFloat(),
        StepCounter(),
        RewardScaling(loc=0, scale=0.01),
    ),
)
env.transform[0].init_stats(num_iter=1000, reduce_dim=0, cat_dim=0)

from torchrl.envs import check_env_specs

check_env_specs(env)

from torchrl.modules import ProbabilisticActor, TanhNormal
from torchrl.modules import NormalParamExtractor

num_cells = 32
device = "cpu"

actor_net = nn.Sequential(
    nn.LazyLinear(num_cells, device=device),
    nn.Tanh(),
    nn.LazyLinear(num_cells, device=device),
    nn.Tanh(),
    nn.LazyLinear(num_cells, device=device),
    nn.Tanh(),
    nn.LazyLinear(2 * env.action_spec.shape[-1], device=device),
    NormalParamExtractor(),
)

policy_module = TensorDictModule(
    actor_net, in_keys=["observation"], out_keys=["loc", "scale"]
)

policy_module = ProbabilisticActor(
    module=policy_module,
    spec=env.action_spec,
    in_keys=["loc", "scale"],
    distribution_class=TanhNormal,
    distribution_kwargs={
        "low": env.action_spec_unbatched.space.low,
        "high": env.action_spec_unbatched.space.high,
    },
    return_log_prob=True,
)

#n_obs = env.observation_spec["observation"].shape[-1]

#create a model: takes observations as inputs, and outputs categorical actions
model = nn.Sequential(
    nn.LazyLinear(num_cells),
    nn.Tanh(),
    nn.LazyLinear(num_cells),
    nn.Tanh(),
    nn.LazyLinear(1),
)

#go from observations to logits using the model
value_net = TensorDictModule(
    model,
    in_keys=["observation"],
    out_keys=["state_value"],          
)

# Initialize lazy layers with a dummy forward pass
dummy_td = env.reset()
policy_module(dummy_td)   # initializes actor lazy layers
value_net(dummy_td)       # initializes critic lazy layers



from torchrl.collectors import SyncDataCollector
from torchrl.data import LazyTensorStorage, ReplayBuffer, SamplerWithoutReplacement
from torchrl.record import CSVLogger
import torch.nn.functional as F

logger = CSVLogger(exp_name="my_exp")

frames_per_batch = 200
total_frames = 1_000_000
gamma = 0.99



from torchrl.objectives import ClipPPOLoss
from torchrl.objectives.value import GAE

loss_fn = ClipPPOLoss(
    actor_network=policy_module,
    critic_network=value_net,
    clip_epsilon=0.2,        # the ε — constrains how much policy can change
    entropy_bonus=True,
    entropy_coeff=0.01,
    critic_coeff=0.5,
    loss_critic_type="l2",
)

adv_module = GAE(
    gamma=0.99,
    lmbda=0.95,
    value_network=value_net,
    average_gae=True,
)

# PPO uses a replay buffer to do multiple epochs over the same batch, though no data is kept between batches
from torchrl.data import LazyTensorStorage, ReplayBuffer, SamplerWithoutReplacement

frames_per_batch = 200   # back to larger batches - PPO handles this fine
num_epochs = 10          # reuse each batch 10 times
minibatch_size = 50      # update on 50-frame minibatches within each epoch

replay_buffer = ReplayBuffer(
    storage=LazyTensorStorage(frames_per_batch),
    sampler=SamplerWithoutReplacement(),  # ensures each sample used once per epoch
    batch_size=20
)

collector = SyncDataCollector(
    env,
    policy_module,
    frames_per_batch=frames_per_batch,
    total_frames=total_frames,
    split_trajs=False,
    device=device,
)

optim = Adam(loss_fn.parameters(), lr=0.0003)  

for data in collector:
    reward = data["next", "reward"].squeeze(-1)

    # Compute GAE advantages once for the whole batch
    with torch.no_grad():
        adv_module(data)

    # Store the batch in the replay buffer
    replay_buffer.extend(data)

    # Now do K epochs of minibatch updates
    for epoch in range(num_epochs):
        for minibatch in replay_buffer:  # SamplerWithoutReplacement covers full batch once
            loss_td = loss_fn(minibatch)
            loss = loss_td["loss_objective"] + loss_td["loss_critic"] + loss_td["loss_entropy"]

            optim.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(loss_fn.parameters(), max_norm=0.5)
            optim.step()

    print(
        f"Reward: {reward.mean():+7.3f} | "
        f"Actor Loss: {loss_td['loss_objective'].item():+7.3f} | "
        f"Critic Loss: {loss_td['loss_critic'].item():7.3f} | "
        f"Entropy: {loss_td['loss_entropy'].item():+7.4f} | "
        f"Clip frac: {loss_td['clip_fraction'].item():.3f}"
    )



2026-03-10 09:48:44,958 [torchrl][INFO]    check_env_specs succeeded! [END]


/home/titan2/data/conda-envs/torchrl_latest/lib/python3.12/site-packages/torchrl/collectors/_base.py:1045: DeprecationWarning: SyncDataCollector has been deprecated and will be removed in v0.13. Please use Collector instead.
  warnings.warn(


2026-03-10 09:48:45,769 [torchrl][INFO]    Initialized LazyTensorStorage with torch.Size([200]) shape [END]
Reward:  -0.084 | Actor Loss:  -0.102 | Critic Loss:   0.054 | Entropy: -0.0144 | Clip frac: 0.050
Reward:  -0.055 | Actor Loss:  -0.033 | Critic Loss:   0.028 | Entropy: -0.0131 | Clip frac: 0.000
Reward:  -0.061 | Actor Loss:  -0.169 | Critic Loss:   0.014 | Entropy: -0.0140 | Clip frac: 0.350
Reward:  -0.083 | Actor Loss:  +0.280 | Critic Loss:   0.003 | Entropy: -0.0129 | Clip frac: 0.100
Reward:  -0.076 | Actor Loss:  -0.025 | Critic Loss:   0.016 | Entropy: -0.0135 | Clip frac: 0.150
Reward:  -0.059 | Actor Loss:  -0.264 | Critic Loss:   0.016 | Entropy: -0.0133 | Clip frac: 0.050
Reward:  -0.065 | Actor Loss:  +0.175 | Critic Loss:   0.014 | Entropy: -0.0133 | Clip frac: 0.300
Reward:  -0.061 | Actor Loss:  +0.118 | Critic Loss:   0.003 | Entropy: -0.0132 | Clip frac: 0.400
Reward:  -0.066 | Actor Loss:  +0.012 | Critic Loss:   0.011 | Entropy: -0.0113 | Clip frac: 0.200
R

KeyboardInterrupt: 

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

loss_value  = [v for _, v in logger.experiment.scalars["loss_value"]]
loss_policy = [v for _, v in logger.experiment.scalars["loss_policy"]]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, loss, title in zip(
    axes,
    [loss_value, loss_policy],
    ["Critic Loss (Value)", "Actor Loss (Policy)"]
):
    smoothed = pd.Series(loss).rolling(window=50).mean()
    ax.plot(loss, alpha=0.3, label="raw")
    ax.plot(smoothed, label="smoothed (50-step avg)")
    ax.set_title(title)
    ax.set_xlabel("Training Step")
    ax.set_ylabel("Loss")
    ax.legend()

plt.tight_layout()
plt.show()

Saving and loading rl models

Testing RL models (for the next notebook)

- show how to save here 
- Next we'll show how to load the model
- run it in deterministic mode on a new environment
- record data and plot reward curves, how to meaninfully compare between curves
- Do an abolition test (try without GAE, or between two different types of architectures, or ... let the students choose their abelation then as a class we can discuss)

